In [3]:
import json
import torch
import os 
import logging

from google.cloud import storage
from transformers import AutoTokenizer, AutoModelForSequenceClassification

/home/lapiceroazul4/Documentos/romboost/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
MODEL_PATH = "/tmp/sentiment-analysis-client"

def download_model_from_gcs(bucket_name, prefix="sentiment-analysis/sentiment-analysis-client"):
    
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    # Descargar el modelo y el tokenizer
    blobs = bucket.list_blobs(prefix=prefix)
    os.makedirs(MODEL_PATH, exist_ok=True)
    for blob in blobs:
        file_path = os.path.join(MODEL_PATH, blob.name.split("/")[-1])
        blob.download_to_filename(file_path)
        print(f"Downloaded {blob.name} to {file_path}")

In [ ]:
# Set the path to your Google Cloud credentials JSON file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../your-credentials.json"

In [ ]:
download_model_from_gcs("bucket", "sentiment-analysis/sentiment-analysis-client")

# Cargar el modelo y el tokenizer desde la ruta temporal
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

Downloaded sentiment-analysis/sentiment-analysis-client/config.json to /tmp/sentiment-analysis-client/config.json
Downloaded sentiment-analysis/sentiment-analysis-client/model.safetensors to /tmp/sentiment-analysis-client/model.safetensors
Downloaded sentiment-analysis/sentiment-analysis-client/special_tokens_map.json to /tmp/sentiment-analysis-client/special_tokens_map.json
Downloaded sentiment-analysis/sentiment-analysis-client/tokenizer_config.json to /tmp/sentiment-analysis-client/tokenizer_config.json
Downloaded sentiment-analysis/sentiment-analysis-client/vocab.txt to /tmp/sentiment-analysis-client/vocab.txt


In [20]:
# Since the model outputs 0-4, we need to map it to 1-5
def map_sentiment(value):
    return value + 1  # 0-4 -> 1-5

In [28]:
def process_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    outputs = model(**inputs)
    sentiment_score = torch.argmax(outputs.logits, dim=1).item()
    return map_sentiment(sentiment_score)

def upload_to_gcs(data, call_id, bucket_name, prefix="sentiment-analysis/results"):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    
    blob = bucket.blob(f"{prefix}/{call_id}.json")
    blob.upload_from_string(json.dumps(data), content_type="application/json")
    logging.info(f"Uploaded {call_id}.json to {bucket_name}.")

In [29]:
test_request = {
        "call_id": "1235",
        "client_name": "client-test",
        "call_duration": "00:15:00",
        "data": {
                "client": "Estoy molesto porque me llaman constantemente, aunque ya he dejado claro que no quiero saber nada de seguros en este momento. Les agradecería que respeten mi decisión y dejen de insistir, ya que no me interesa y prefiero no recibir más llamadas.",
                "agent": "agent-message",
            }
        }

In [30]:
client_name = test_request.get("client_name")
client_message = test_request["data"].get("client")
agent_message = test_request["data"].get("agent")

In [31]:
client_sentiment = process_sentiment(client_message)
agent_sentiment = process_sentiment(agent_message)

In [ ]:
example_result = {
            "call_id": "12232",
            "client_name": "client_name",
            "call_duration": "00:15",
            "data": {
                "client": client_message,
                "client_sentiment": client_sentiment,
                "agent": agent_message,
                "agent_sentiment": agent_sentiment
            }
        }

In [33]:
print(result)

{'call_id': '12232', 'client_name': 'client_name', 'call_duration': '00:15', 'data': {'client': 'Estoy molesto porque me llaman constantemente, aunque ya he dejado claro que no quiero saber nada de seguros en este momento. Les agradecería que respeten mi decisión y dejen de insistir, ya que no me interesa y prefiero no recibir más llamadas.', 'client_sentiment': 1, 'agent': 'agent-message', 'agent_sentiment': 3}}


In [34]:
upload_to_gcs(result, client_name, "madrinas-bucket", "sentiment-analysis/results")
logging.info(f"Successfully processed request for call_id: {client_name}")